# Three-threshold directional-change mechanics

This notebook studies `GBPUSD.r` with three independent directional-change clocks:

- small: 0.20%
- medium: 0.40%
- large: 0.60%

The objective is structural intuition, not prediction or trading. We will inspect the three event ledgers, causal live modes, all eight directional configurations, legal transitions, adjacent-scale propagation attempts, and the nesting of lower-scale DCCs inside higher-scale intervals.

The threshold ratios differ: medium/small is 2.0, while large/medium is 1.5. Exact propagation frequencies and durations should therefore not be expected to match across the two hierarchy edges.

In [1]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from dc_basics.data import resolve_data_root
from dc_basics.core import dc_live_features, parse_dc_events
from dc_basics.plotting import (
    plot_three_threshold_directional_change,
    plot_three_threshold_modes,
)

DATA_ROOT = resolve_data_root()  # $DC_DATA_ROOT or .dc_data_root
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
SMALL_THETA = 0.002
MEDIUM_THETA = 0.004
LARGE_THETA = 0.006
N_PRICES = 10_000

if not 0.0 < SMALL_THETA < MEDIUM_THETA < LARGE_THETA:
    raise ValueError("Thresholds must satisfy 0 < small < medium < large.")

print(f"Symbol: {SYMBOL}")
print(f"Thresholds: {SMALL_THETA:.2%}, {MEDIUM_THETA:.2%}, {LARGE_THETA:.2%}")
print(f"Ratios: medium/small={MEDIUM_THETA / SMALL_THETA:.2f}, "
      f"large/medium={LARGE_THETA / MEDIUM_THETA:.2f}")

Symbol: GBPUSD.r
Thresholds: 0.20%, 0.40%, 0.60%
Ratios: medium/small=2.00, large/medium=1.50


In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"


from dc_basics.enums import DcMode


class ModeConfiguration(StrEnum):
    UP_UP_UP = "up/up/up"
    UP_UP_DOWN = "up/up/down"
    UP_DOWN_UP = "up/down/up"
    UP_DOWN_DOWN = "up/down/down"
    DOWN_UP_UP = "down/up/up"
    DOWN_UP_DOWN = "down/up/down"
    DOWN_DOWN_UP = "down/down/up"
    DOWN_DOWN_DOWN = "down/down/down"


class HierarchyShape(StrEnum):
    BROAD_AGREEMENT = "broad_agreement"
    SMALL_COUNTER_MOVE = "small_counter_move"
    PROPAGATED_TO_MEDIUM = "propagated_to_medium"
    FRAGMENTED_MEDIUM = "fragmented_medium"


from dc_basics.enums import AttemptOutcome


class TransitionSignature(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    SMALL_MEDIUM = "small+medium"
    SMALL_LARGE = "small+large"
    MEDIUM_LARGE = "medium+large"
    ALL = "small+medium+large"


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE)
SCALE_SETTINGS = {
    Scale.SMALL: {"prefix": "S", "theta": SMALL_THETA},
    Scale.MEDIUM: {"prefix": "M", "theta": MEDIUM_THETA},
    Scale.LARGE: {"prefix": "L", "theta": LARGE_THETA},
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}
CONFIGURATION_BY_MODES = {
    tuple(configuration.value.split("/")): configuration.value
    for configuration in ModeConfiguration
}
SIGNATURE_BY_CHANGED_SCALES = {
    (True, False, False): TransitionSignature.SMALL.value,
    (False, True, False): TransitionSignature.MEDIUM.value,
    (False, False, True): TransitionSignature.LARGE.value,
    (True, True, False): TransitionSignature.SMALL_MEDIUM.value,
    (True, False, True): TransitionSignature.SMALL_LARGE.value,
    (False, True, True): TransitionSignature.MEDIUM_LARGE.value,
    (True, True, True): TransitionSignature.ALL.value,
}

## Information contract

Each threshold is processed independently from left to right. A live mode is known on its bar. A completed event's ending extreme, TMV, T, and R become known only at its DCC, so every completed-event row receives `information_time = dcc_idx`.

Configuration order is always `(small, medium, large)`. For example, `down/down/up` means the small and medium modes agree downward while the large mode remains upward. It does not mean that a large downward DCC has already occurred.

## Load and validate GBPUSD closes

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no 'close' column.")
prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


## Build three independent event clocks

Event identifiers are local to their scale. `S10`, `M10`, and `L10` are unrelated ledger row numbers. No event is matched across thresholds merely because its identifier has the same number.

In [4]:
from dc_basics.events import prepare_event_ledger


event_ledgers = {}
live_frames = {}
for scale in SCALE_ORDER:
    settings = SCALE_SETTINGS[scale]
    raw_events = parse_dc_events(prices=prices, theta=settings["theta"])
    event_ledgers[scale] = prepare_event_ledger(
        source=raw_events,
        prefix=settings["prefix"],
        scale=scale,
        theta=settings["theta"],
    )
    live_frames[scale] = dc_live_features(prices, theta=settings["theta"])

In [5]:
event_summary_records = []
for scale in SCALE_ORDER:
    ledger = event_ledgers[scale]
    event_spacing = np.diff(ledger["dcc_pos"].to_numpy(dtype=np.int64))
    event_summary_records.append({
        "scale": scale.value,
        "threshold": SCALE_SETTINGS[scale]["theta"],
        "completed_events": len(ledger),
        "median_tmv": ledger["tmv"].median(),
        "median_t": ledger["t"].median(),
        "median_r": ledger["r"].median(),
        "median_bars_between_dccs": np.median(event_spacing),
    })
event_summary = pd.DataFrame(event_summary_records).set_index("scale")
event_summary.style.format({
    "threshold": "{:.2%}",
    "completed_events": "{:,}",
    "median_tmv": "{:.4f}",
    "median_t": "{:.1f}",
    "median_r": "{:.3e}",
    "median_bars_between_dccs": "{:.1f}",
})

,threshold,completed_events,median_tmv,median_t,median_r,median_bars_between_dccs
scale,,,,,,
small,0.20%,"2,697",1.8584,60.0,6.655e-05,60.0
medium,0.40%,839,1.7465,229.0,3.256e-05,227.5
large,0.60%,392,1.7182,473.5,2.413e-05,448.0


## Price with all three completed-event ledgers

Small legs are thin and solid, medium legs are dotted, and large legs are thick and dashed. Each scale-direction pair also has a distinct colour. The markers at past extremes are retrospective locations; their event information becomes available only at the corresponding DCC marker.

In [6]:
event_overlay = plot_three_threshold_directional_change(
    prices=prices,
    small_events=event_ledgers[Scale.SMALL],
    medium_events=event_ledgers[Scale.MEDIUM],
    large_events=event_ledgers[Scale.LARGE],
    small_theta=SMALL_THETA,
    medium_theta=MEDIUM_THETA,
    large_theta=LARGE_THETA,
    n_prices=N_PRICES,
)
event_overlay.show()

## Construct the causally known three-mode path

The configuration is emitted only after all three parsers have initialized. From then onward it is known on every bar and changes only when at least one threshold confirms a DCC.

In [7]:
mode_frame = pd.DataFrame(index=prices.index)
for scale in SCALE_ORDER:
    live = live_frames[scale]
    ledger = event_ledgers[scale]
    mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
    mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(
        ledger.set_index("dcc_idx")["event_id"]
    )

mode_columns = [f"{scale.value}_mode" for scale in SCALE_ORDER]
all_modes_known = mode_frame[mode_columns].notna().all(axis=1)
known_mode_values = mode_frame.loc[all_modes_known, mode_columns]
mode_frame["configuration"] = pd.Series(pd.NA, index=mode_frame.index, dtype="string")
mode_frame.loc[all_modes_known, "configuration"] = pd.Series(
    [CONFIGURATION_BY_MODES[tuple(row)] for row in known_mode_values.itertuples(index=False, name=None)],
    index=known_mode_values.index,
    dtype="string",
)

small_mode = mode_frame["small_mode"]
medium_mode = mode_frame["medium_mode"]
large_mode = mode_frame["large_mode"]
mode_frame["small_medium_relationship"] = pd.Series(pd.NA, index=mode_frame.index, dtype="string")
mode_frame["medium_large_relationship"] = pd.Series(pd.NA, index=mode_frame.index, dtype="string")
small_medium_known = small_mode.notna() & medium_mode.notna()
medium_large_known = medium_mode.notna() & large_mode.notna()
mode_frame.loc[small_medium_known, "small_medium_relationship"] = np.where(
    small_mode.loc[small_medium_known].eq(medium_mode.loc[small_medium_known]),
    "agreement",
    "disagreement",
)
mode_frame.loc[medium_large_known, "medium_large_relationship"] = np.where(
    medium_mode.loc[medium_large_known].eq(large_mode.loc[medium_large_known]),
    "agreement",
    "disagreement",
)

mode_frame["hierarchy_shape"] = pd.Series(pd.NA, index=mode_frame.index, dtype="string")
mode_frame.loc[all_modes_known & small_mode.eq(medium_mode) & medium_mode.eq(large_mode), "hierarchy_shape"] = (
    HierarchyShape.BROAD_AGREEMENT.value
)
mode_frame.loc[all_modes_known & small_mode.ne(medium_mode) & medium_mode.eq(large_mode), "hierarchy_shape"] = (
    HierarchyShape.SMALL_COUNTER_MOVE.value
)
mode_frame.loc[all_modes_known & small_mode.eq(medium_mode) & medium_mode.ne(large_mode), "hierarchy_shape"] = (
    HierarchyShape.PROPAGATED_TO_MEDIUM.value
)
mode_frame.loc[all_modes_known & small_mode.eq(large_mode) & medium_mode.ne(large_mode), "hierarchy_shape"] = (
    HierarchyShape.FRAGMENTED_MEDIUM.value
)

if mode_frame.loc[all_modes_known, ["configuration", "hierarchy_shape"]].isna().any().any():
    raise AssertionError("A fully initialized bar lacks a configuration or hierarchy shape.")

initialization_audit = pd.DataFrame({
    "first_mode_time": [live_frames[scale]["direction"].first_valid_index() for scale in SCALE_ORDER],
    "completed_events": [len(event_ledgers[scale]) for scale in SCALE_ORDER],
}, index=[scale.value for scale in SCALE_ORDER])
initialization_audit.loc["all_scales", "first_mode_time"] = mode_frame.index[all_modes_known][0]
initialization_audit

,first_mode_time,completed_events
small,2023-05-22 16:15:00+02:00,2697.0
medium,2023-05-22 17:50:00+02:00,839.0
large,2023-05-23 11:30:00+02:00,392.0
all_scales,2023-05-23 11:30:00+02:00,NaN


In [8]:
mode_ribbons = plot_three_threshold_modes(
    prices=prices,
    mode_frame=mode_frame,
    n_prices=N_PRICES,
)
mode_ribbons.show()

## Eight configurations, four structural shapes

Direction doubles each structural shape into an up/down version:

- `up/up/up`, `down/down/down`: broad agreement;
- `down/up/up`, `up/down/down`: a small-only counter-move;
- `down/down/up`, `up/up/down`: propagation has reached medium but not large;
- `up/down/up`, `down/up/down`: the medium mode opposes both outer modes, a fragmented/non-monotone hierarchy.

A shape is a causal description of the current clocks, not a future outcome label.

In [9]:
state_path = mode_frame.loc[all_modes_known, ["configuration", "hierarchy_shape", *mode_columns]].copy()
state_path = state_path.reset_index(names="time")
state_path["bar_position"] = prices.index.get_indexer(state_path["time"])
state_path["run_id"] = (
    state_path["configuration"].ne(state_path["configuration"].shift()).fillna(True).cumsum()
)
state_runs = state_path.groupby("run_id", sort=True).agg(
    configuration=("configuration", "first"),
    hierarchy_shape=("hierarchy_shape", "first"),
    start_time=("time", "first"),
    end_time=("time", "last"),
    start_position=("bar_position", "first"),
    end_position=("bar_position", "last"),
    bars=("configuration", "size"),
).reset_index(drop=True)
# The final run is still in progress at the last bar: its length is right-censored.
state_runs["right_censored"] = state_runs["end_position"].eq(len(prices) - 1)

configuration_order = [configuration.value for configuration in ModeConfiguration]
configuration_summary = state_runs.groupby("configuration", sort=False).agg(
    runs=("configuration", "size"),
    median_run_bars=("bars", "median"),
    p90_run_bars=("bars", lambda values: values.quantile(0.90)),
)
configuration_summary["bars"] = state_path["configuration"].value_counts()
configuration_summary = configuration_summary.reindex(configuration_order)
configuration_summary[["runs", "bars"]] = configuration_summary[["runs", "bars"]].fillna(0).astype(int)
configuration_summary["bar_share"] = configuration_summary["bars"] / len(state_path)
configuration_summary.style.format({
    "runs": "{:,}",
    "median_run_bars": "{:.1f}",
    "p90_run_bars": "{:.1f}",
    "bars": "{:,}",
    "bar_share": "{:.2%}",
})

,runs,median_run_bars,p90_run_bars,bars,bar_share
configuration,,,,,
up/up/up,648,60.0,228.3,"61,359",23.75%
up/up/down,418,34.0,152.0,"24,878",9.63%
up/down/up,251,34.0,151.0,"14,200",5.50%
up/down/down,618,36.5,143.6,"35,550",13.76%
down/up/up,625,36.0,157.0,"37,038",14.34%
down/up/down,229,30.0,140.6,"12,176",4.71%
down/down/up,441,30.0,135.0,"23,322",9.03%
down/down/down,631,48.0,202.0,"49,792",19.28%


In [10]:
shape_summary = state_runs.groupby("hierarchy_shape", sort=False).agg(
    runs=("hierarchy_shape", "size"),
    median_run_bars=("bars", "median"),
    p90_run_bars=("bars", lambda values: values.quantile(0.90)),
)
shape_summary["bars"] = state_path["hierarchy_shape"].value_counts()
shape_summary["bar_share"] = shape_summary["bars"] / len(state_path)
display(shape_summary.style.format({
    "runs": "{:,}",
    "median_run_bars": "{:.1f}",
    "p90_run_bars": "{:.1f}",
    "bars": "{:,}",
    "bar_share": "{:.2%}",
}))

configuration_colors = [
    "#166534", "#2563eb", "#a21caf", "#f59e0b",
    "#16a34a", "#7c3aed", "#0f766e", "#991b1b",
]
configuration_occupancy_fig = go.Figure(go.Bar(
    x=configuration_summary.index,
    y=configuration_summary["bar_share"],
    marker={"color": configuration_colors},
    customdata=configuration_summary[["runs", "median_run_bars"]].to_numpy(),
    hovertemplate=(
        "configuration=%{x}<br>bar share=%{y:.2%}<br>runs=%{customdata[0]:,.0f}"
        "<br>median run=%{customdata[1]:.1f} bars<extra></extra>"
    ),
))
configuration_occupancy_fig.update_layout(
    title="Causal occupancy of all eight mode configurations",
    template="plotly_white",
    height=480,
    xaxis_title="Small / medium / large mode",
    yaxis_title="Share of fully initialized bars",
    yaxis_tickformat=".0%",
)
configuration_occupancy_fig.show()

,runs,median_run_bars,p90_run_bars,bars,bar_share
hierarchy_shape,,,,,
broad_agreement,"1,279",54.0,210.2,"111,151",43.03%
small_counter_move,"1,243",36.0,155.6,"72,588",28.10%
propagated_to_medium,859,32.0,145.4,"48,200",18.66%
fragmented_medium,480,32.0,143.2,"26,376",10.21%


## Legal configuration transitions and same-bar confirmations

A configuration may change only when the corresponding scale confirms a DCC on that bar. One-scale transitions are directly ordered. Multi-scale transitions are only known together at the close; close-only data cannot recover their intrabar ordering.

In [11]:
previous_configuration = state_path["configuration"].shift()
is_transition = state_path["configuration"].ne(previous_configuration) & previous_configuration.notna()
transitions = state_path.loc[
    is_transition, ["time", "bar_position", "configuration", "hierarchy_shape", *mode_columns]
].copy()
transitions["previous_configuration"] = previous_configuration.loc[is_transition].to_numpy()
transitions["previous_shape"] = state_path["hierarchy_shape"].shift().loc[is_transition].to_numpy()
changed_columns = []
for scale in SCALE_ORDER:
    mode_column = f"{scale.value}_mode"
    changed_column = f"{scale.value}_changed"
    transitions[changed_column] = (
        state_path.loc[is_transition, mode_column].to_numpy()
        != state_path[mode_column].shift().loc[is_transition].to_numpy()
    )
    changed_columns.append(changed_column)
    dcc_columns = [f"{scale.value}_upward_dcc", f"{scale.value}_downward_dcc"]
    observed_dcc = mode_frame.loc[pd.DatetimeIndex(transitions["time"]), dcc_columns].any(axis=1).to_numpy()
    if not np.array_equal(transitions[changed_column].to_numpy(), observed_dcc):
        raise AssertionError(f"A {scale.value} mode transition disagrees with its DCC flags.")

transitions["changed_scale_count"] = transitions[changed_columns].sum(axis=1)
transitions["transition_signature"] = [
    SIGNATURE_BY_CHANGED_SCALES[tuple(row)]
    for row in transitions[changed_columns].itertuples(index=False, name=None)
]
transition_summary = transitions.groupby("transition_signature", sort=False).agg(
    transitions=("configuration", "size"),
)
transition_summary["share"] = transition_summary["transitions"] / len(transitions)
transition_matrix = pd.crosstab(
    transitions["previous_configuration"],
    transitions["configuration"],
    rownames=["from"],
    colnames=["to"],
)
display(transition_summary.style.format({"transitions": "{:,}", "share": "{:.2%}"}))
display(transition_matrix)

,transitions,share
transition_signature,,
small,"2,644",68.50%
medium,788,20.41%
large,370,9.59%
small+medium,36,0.93%
small+large,7,0.18%
small+medium+large,8,0.21%
medium+large,7,0.18%


to,down/down/down,down/down/up,down/up/down,down/up/up,up/down/down,up/down/up,up/up/down,up/up/up
from,,,,,,,,
down/down/down,0,0,0,0,618,0,10,3
down/down/up,185,0,0,0,0,251,0,5
down/up/down,107,0,0,0,0,0,118,4
down/up/up,3,283,0,0,0,0,0,339
up/down/down,324,0,0,0,0,0,290,4
up/down/up,3,140,0,0,0,0,0,108
up/up/down,3,0,229,0,0,0,0,185
up/up/up,5,18,0,625,0,0,0,0


In [12]:
same_bar_transitions = transitions.loc[transitions["changed_scale_count"].gt(1)].copy()
same_bar_audit = same_bar_transitions.groupby("transition_signature", sort=False).agg(
    bars=("time", "size"),
    first_time=("time", "min"),
    last_time=("time", "max"),
)
print(f"All configuration transitions: {len(transitions):,}")
print(f"Multi-scale same-bar transitions: {len(same_bar_transitions):,}")
same_bar_audit

All configuration transitions: 3,860
Multi-scale same-bar transitions: 58


,bars,first_time,last_time
transition_signature,,,
small+medium,36,2023-07-18 13:00:00+02:00,2026-10-28 20:40:00+02:00
small+large,7,2023-08-25 17:00:00+02:00,2025-07-09 17:50:00+02:00
small+medium+large,8,2023-10-06 15:30:00+02:00,2026-03-23 16:05:00+02:00
medium+large,7,2023-12-20 09:00:00+02:00,2026-04-13 02:00:00+02:00


## Adjacent-scale propagation contests

We now run the familiar two-threshold state machine independently on each adjacent edge:

1. small against medium;
2. medium against large.

An attempt begins when the lower scale confirms a mode against the upper scale. It propagates if the upper DCC joins first and is rejected if the lower scale flips back first. Outcome and resolution duration are hindsight until the resolution bar. These edge tables do not yet claim that one uniquely identified move traversed both edges.

In [13]:
from dc_basics.hierarchy import build_adjacent_attempts


In [14]:
adjacent_attempts = pd.concat([
    build_adjacent_attempts(mode_frame, Scale.SMALL, Scale.MEDIUM),
    build_adjacent_attempts(mode_frame, Scale.MEDIUM, Scale.LARGE),
], ignore_index=True)
adjacent_attempts["outcome_information_time"] = adjacent_attempts["resolution_time"]
adjacent_attempts["outcome_is_hindsight_at_start"] = (
    adjacent_attempts["outcome_information_time"].gt(adjacent_attempts["start_time"])
)

attempt_outcome_audit = adjacent_attempts.groupby(["edge", "outcome"], sort=False).agg(
    attempts=("initiating_event_id", "size"),
    median_resolution_bars=("resolution_bars", "median"),
)
clean_attempts = adjacent_attempts.loc[
    adjacent_attempts["outcome"].isin([AttemptOutcome.PROPAGATED.value, AttemptOutcome.REJECTED.value])
    & adjacent_attempts["resolution_bars"].gt(0)
].copy()
if not clean_attempts["outcome_is_hindsight_at_start"].all():
    raise AssertionError("A clean adjacent-scale outcome was already known at initiation.")

edge_records = []
for edge, edge_attempts in adjacent_attempts.groupby("edge", sort=False):
    edge_clean = clean_attempts.loc[clean_attempts["edge"].eq(edge)]
    edge_records.append({
        "edge": edge,
        "all_attempts": len(edge_attempts),
        "same_bar_propagations": int(edge_attempts["resolution_bars"].eq(0).sum()),
        "clean_future_attempts": len(edge_clean),
        "clean_propagation_rate": edge_clean["outcome"].eq(AttemptOutcome.PROPAGATED.value).mean(),
        "median_clean_resolution_bars": edge_clean["resolution_bars"].median(),
    })
edge_summary = pd.DataFrame(edge_records).set_index("edge")
display(attempt_outcome_audit.style.format({
    "attempts": "{:,}",
    "median_resolution_bars": "{:.1f}",
}))
display(edge_summary.style.format({
    "all_attempts": "{:,}",
    "same_bar_propagations": "{:,}",
    "clean_future_attempts": "{:,}",
    "clean_propagation_rate": "{:.2%}",
    "median_clean_resolution_bars": "{:.1f}",
}))

D:\projects\quant-studies\signalfoundry\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,all_attempts,same_bar_propagations,clean_future_attempts,clean_propagation_rate,median_clean_resolution_bars
edge,,,,,
small->medium,"1,768",44,"1,724",46.11%,35.0
medium->large,616,15,600,62.83%,72.0


## Event nesting between adjacent thresholds

Between two consecutive upper-scale DCCs, the number of lower-scale DCCs is almost always positive and odd. Every pair before the terminal odd event represents one rejected lower-scale counter-attempt. The interval result becomes known only at the ending upper DCC.

This is a near-invariant, not a law. Both thresholds are multiplicative and are measured from *different* running extremes, so on rare paths the upper clock confirms a reversal while the lower clock still points the other way (the relative window is roughly the square of the small threshold). Such an interval holds an even number of lower DCCs. `build_nesting_intervals` keeps it, flags it with `parity_violation`, and leaves `prior_rejections` missing; the audit below counts them, and `find_upper_led_reversals` lists the upper DCCs that came first.

In [15]:
from dc_basics.hierarchy import (
    build_nesting_intervals,
    find_upper_led_reversals,
    summarize_nesting_parity,
)


nesting_intervals = pd.concat([
    build_nesting_intervals(
        event_ledgers[Scale.SMALL], event_ledgers[Scale.MEDIUM], Scale.SMALL, Scale.MEDIUM
    ),
    build_nesting_intervals(
        event_ledgers[Scale.MEDIUM], event_ledgers[Scale.LARGE], Scale.MEDIUM, Scale.LARGE
    ),
], ignore_index=True)

upper_led_reversals = pd.concat([
    find_upper_led_reversals(mode_frame, Scale.SMALL, Scale.MEDIUM),
    find_upper_led_reversals(mode_frame, Scale.MEDIUM, Scale.LARGE),
], ignore_index=True)
nesting_parity_audit = summarize_nesting_parity(nesting_intervals)
nesting_parity_audit["upper_led_reversals"] = (
    upper_led_reversals["edge"].value_counts().reindex(nesting_parity_audit.index, fill_value=0)
)
display(nesting_parity_audit.style.format({"violation_share": "{:.3%}"}))
if not upper_led_reversals.empty:
    display(upper_led_reversals)


,upper_intervals,parity_violations,empty_intervals,violation_share,upper_led_reversals
edge,,,,,
small->medium,838,0,0,0.000%,0
medium->large,391,0,0,0.000%,0


In [16]:
nesting_summary = nesting_intervals.groupby(["edge", "upper_completed_direction"], sort=False).agg(
    upper_intervals=("upper_event_id", "size"),
    lower_dcc_count_median=("lower_dcc_count", "median"),
    lower_dcc_count_p90=("lower_dcc_count", lambda values: values.quantile(0.90)),
    prior_rejections_median=("prior_rejections", "median"),
    prior_rejections_p90=("prior_rejections", lambda values: values.quantile(0.90)),
    interval_bars_median=("interval_bars", "median"),
)
nesting_summary.style.format({
    "upper_intervals": "{:,}",
    "lower_dcc_count_median": "{:.1f}",
    "lower_dcc_count_p90": "{:.1f}",
    "prior_rejections_median": "{:.1f}",
    "prior_rejections_p90": "{:.1f}",
    "interval_bars_median": "{:.1f}",
})

## Causal configuration casebook

The default example selects the most recent *completed* run where small and medium agree against large. A run still in progress at the last bar is right-censored (its end and length are unknown), so it is excluded. The shaded interval is identified from causally known mode states. Change `CASEBOOK_SHAPE` to inspect the other structural shapes.

In [17]:
CASEBOOK_SHAPE = HierarchyShape.PROPAGATED_TO_MEDIUM
CASEBOOK_BARS_BEFORE = 150
CASEBOOK_BARS_AFTER = 150
shape_examples = state_runs.loc[
    state_runs["hierarchy_shape"].eq(CASEBOOK_SHAPE.value) & ~state_runs["right_censored"]
]
if shape_examples.empty:
    raise ValueError(f"No completed {CASEBOOK_SHAPE.value} run exists in this sample.")
example = shape_examples.iloc[-1]
window_start_position = max(0, int(example["start_position"]) - CASEBOOK_BARS_BEFORE)
window_end_position = min(len(prices), int(example["end_position"]) + CASEBOOK_BARS_AFTER + 1)
case_prices = prices.iloc[window_start_position:window_end_position]
casebook_fig = plot_three_threshold_directional_change(
    prices=case_prices,
    small_events=event_ledgers[Scale.SMALL],
    medium_events=event_ledgers[Scale.MEDIUM],
    large_events=event_ledgers[Scale.LARGE],
    small_theta=SMALL_THETA,
    medium_theta=MEDIUM_THETA,
    large_theta=LARGE_THETA,
    n_prices=len(case_prices),
    title=f"{SYMBOL}: {CASEBOOK_SHAPE.value} ({example['configuration']})",
)
casebook_fig.add_vrect(
    x0=example["start_time"],
    x1=example["end_time"],
    fillcolor="#a78bfa",
    opacity=0.15,
    line_width=1,
    line_dash="dot",
)
display(example[["configuration", "hierarchy_shape", "start_time", "end_time", "bars"]])
casebook_fig.show()

configuration                   down/down/up
hierarchy_shape         propagated_to_medium
start_time         2026-11-05 23:05:00+02:00
end_time           2026-11-06 08:40:00+02:00
bars                                     116
Name: 3855, dtype: object

# Three-threshold mechanics completion gate

Before studying event characteristics or probabilities, be able to answer these questions from the outputs:

1. Why are the three event clocks independent even though they use the same price path?
2. Can you read every configuration in `(small, medium, large)` order without confusing a completed leg with the new live mode?
3. Which configurations mean broad agreement, a small pullback, propagation to medium, and medium fragmentation?
4. Which exact DCC changes each component of a configuration?
5. Why is the intrabar order of a multi-scale same-bar transition unknowable from close-only bars?
6. How many small DCCs occur inside a medium interval, and how many medium DCCs occur inside a large interval?
7. Why are nesting counts almost always odd, how do they encode rejected attempts, and what does a `parity_violation` (the upper clock confirming first) mean?
8. Can you distinguish a small-to-medium contest from a medium-to-large contest?
9. At what bar does each adjacent-scale propagated/rejected outcome become known?
10. Why should the two edges differ when their threshold ratios are 2.0 and 1.5?

No HMM, classifier, probability model, or trading rule belongs in this notebook. The next step is to interpret these mechanical outputs and actual price paths before constructing a linked three-scale propagation ladder.